[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Peewee, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)

# prefetch and Load &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup: the catalog's models, four authors with three books each,
and a database that records what each query carried. Run it first, then the tasks in any order.


In [1]:
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

try:
    if version("peewee") != "4.5.1":                                # Colab has 4.4.0, whose wording differs
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "peewee==4.5.1"], check=True)

import peewee
from peewee import (PREFETCH_TYPE, CharField, ForeignKeyField, IntegerField, Load, Model,
                    OperationalError, SqliteDatabase, chunked, prefetch)
from playhouse.test_utils import count_queries

AUTHORS = [                                                         # name, the year of the first book
    ("Ursula Vance", 2014),
    ("Marco Pietra", 2009),
    ("Ines O'Brien", 1998),
    ("Kofi Mensah", 2015),
]

BOOKS = [                                                           # title, author, year, pages
    ("The Salt Road", "Ursula Vance", 2014, 312),
    ("Nightjar", "Ursula Vance", 2018, 244),
    ("The Quiet Engine", "Ursula Vance", 2021, 398),
    ("Stone and Tide", "Marco Pietra", 2009, 501),
    ("The Lantern Keeper", "Marco Pietra", 2016, 276),
    ("Riverwork", "Marco Pietra", 2022, 189),
    ("A Careful Fire", "Ines O'Brien", 1998, 420),
    ("The Long Field", "Ines O'Brien", 2004, 355),
    ("Winter Harbour", "Ines O'Brien", 2011, 263),
    ("The Drum Line", "Kofi Mensah", 2015, 198),
    ("Harmattan", "Kofi Mensah", 2019, 331),
    ("Small Machines", "Kofi Mensah", 2023, 287),
]

class RecordingSqlite(SqliteDatabase):
    """A database that keeps each statement sent through it, and how many values it carried."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.statements = []

    def execute_sql(self, sql, params=None):
        self.statements.append((" ".join(sql.split()), len(params or ())))
        return super().execute_sql(sql, params)

def bound(database, work):
    """Run some work and report the queries it sent and the values those queries carried."""
    database.statements.clear()
    work()
    return len(database.statements), sum(count for _, count in database.statements)

db = RecordingSqlite(":memory:", pragmas={"foreign_keys": 1})       # the guide's database line


class CatalogModel(Model):
    """Every model in the catalog names the database once, here."""

    class Meta:
        database = db


class Author(CatalogModel):
    name = CharField(max_length=60, unique=True)
    first_book = IntegerField()


class Book(CatalogModel):
    title = CharField(max_length=80)
    author = ForeignKeyField(Author, backref="books")
    year = IntegerField(index=True)
    pages = IntegerField()

def build(database):
    """Create the tables and load the catalog, in one transaction."""
    database.create_tables([Author, Book])
    with database.atomic():
        Author.insert_many([{"name": name, "first_book": year} for name, year in AUTHORS]).execute()
        written = {author.name: author.id for author in Author.select()}
        Book.insert_many([{"title": title, "author": written[author], "year": year, "pages": pages}
                          for title, author, year, pages in BOOKS]).execute()


build(db)
print("peewee", peewee.__version__, "|", Author.select().count(), "authors and",
      Book.select().count(), "books, three each")


peewee 4.5.1 | 4 authors and 12 books, three each


**1.** The loop, counted both ways.


In [2]:
with count_queries() as counter:
    for author in Author.select():
        [written.title for written in author.books]
lazy = counter.count

with count_queries() as counter:
    for author in Author.select().with_related(Load(Author.books)):
        [written.title for written in author.books]

print("a loop over author.books:", lazy, "queries")
print("with_related(Load(...)): ", counter.count, "queries")


a loop over author.books: 5 queries
with_related(Load(...)):  2 queries


One query for the authors and one for each of the four, against one for the authors and one for all
their books together.


**2.** The same with `prefetch`, and what `author.books` becomes.


In [3]:
with count_queries() as counter:
    fetched = list(prefetch(Author.select(), Book.select()))
print("prefetch:", counter.count, "queries")

first = fetched[0]
print("type of author.books:", type(first.books).__name__, "|", len(first.books), "books")

with count_queries() as counter:
    [written.title for written in first.books]
print("reading it:", counter.count, "queries")


prefetch: 2 queries
type of author.books: list | 3 books
reading it: 0 queries


A list rather than a query, which is the difference that makes the loop free. On an author that was
not loaded this way, the same attribute would be a `ModelSelect` and reading it would send a query.


**3.** Each author's longest book.


In [4]:
longest_first = Book.select().order_by(Book.pages.desc())

for author in Author.select().order_by(Author.name).with_related(
        Load(Author.books, longest_first, per_parent=1)):
    written = author.books[0]
    print(f"  {author.name:<15} {written.title} ({written.pages} pages)")


  Ines O'Brien    A Careful Fire (420 pages)
  Kofi Mensah     Harmattan (331 pages)
  Marco Pietra    Stone and Tide (501 pages)
  Ursula Vance    The Quiet Engine (398 pages)


`per_parent=1` keeps the first child of each parent, and the child query's `order_by` decides which
one that is. The other eight books were never fetched.


**4.** Only the older books.


In [5]:
older = Book.select().where(Book.year < 2010).order_by(Book.year)

for author in Author.select().order_by(Author.name).with_related(Load(Author.books, older)):
    print(f"  {author.name:<15} {[written.title for written in author.books]}")


  Ines O'Brien    ['A Careful Fire', 'The Long Field']
  Kofi Mensah     []
  Marco Pietra    ['Stone and Tide']
  Ursula Vance    []


Two authors come back with an empty list. A parent with no matching child is still a parent, so it
is returned with nothing attached rather than dropped, which is what makes this different from a
join on the same condition.


**5.** What each strategy binds.


In [6]:
for name in ("WHERE", "JOIN", "MATERIALIZE"):
    db.statements.clear()
    list(Author.select().with_related(
        Load(Author.books, strategy=getattr(PREFETCH_TYPE, name))))
    values = next(count for line, count in db.statements if '"book"' in line)
    print(f"  {name:<12} {values} values bound, for {Author.select().count()} authors")


  WHERE        0 values bound, for 4 authors
  JOIN         0 values bound, for 4 authors
  MATERIALIZE  4 values bound, for 4 authors


`MATERIALIZE` is the one that grows: it binds one value per parent, so the number above is the
number of authors and would be the number of rows in any larger table. The other two send the
parent query again instead of its results, so they bind nothing however many parents there are.


**6.** The loading that is quietly dropped.


In [7]:
rows = list(Author.select().dicts().with_related(Load(Author.books)))

print("keys on a row:", sorted(rows[0]))
print("a books key:", "books" in rows[0])
print("rows returned:", len(rows), "| no exception was raised")


keys on a row: ['first_book', 'id', 'name']
a books key: False
rows returned: 4 | no exception was raised


The keys are the author's own columns and nothing else. `dicts()` and `Load` are each doing what
they say, and together they do not do what the code looks like it asks for. If the related rows are
wanted, the parent query has to return model instances.


---

&#8592; **Back to:** [prefetch and Load](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/07-prefetch-and-load.ipynb)  &nbsp;&middot;&nbsp;  [Peewee, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)
